# 권현성의 BM25·의미 검색 앙상블 실습

- **핵심 개념:** 키워드 검색, 임베딩 검색, RRF 앙상블, 하이브리드 RAG
- **나의 수정:** 청크 중첩과 검색 가중치를 조정하고 근거 중심 답변 프롬프트로 변경
- API 키는 코드에 저장하지 않고 Colab `Secrets`의 `OPENAI_API_KEY`를 사용합니다.


BM25 알고리즘은 사용자가 특정 단어를 검색했을 때, 어떤 문서가 더 적합한지 문서의 순위를 결정하는 알고리즘입니다.

**단어 빈도**  
사용자가 「인공지능」이라는 단어를 검색했다고 생각해봅시다. 문서가 두 개 있는데, 첫 번째 문서에는 「인공지능」이 여러 번 반복해서 등장하고, 두 번째 문서에는 딱 한 번만 등장했다고 합시다. 그렇다면 우리는 첫 번째 문서가 더 관련이 깊다고 생각하게 됩니다. BM25 알고리즘 역시 이러한 판단을 합니다. 이것을 **단어 빈도(Term Frequency, TF)**라고 합니다. 특정 단어가 문서 내에 더 자주 나타날수록 해당 문서에서 그 단어는 중요한 역할을 한다고 판단합니다.

**역문서 빈도**  
하지만 여기서 끝이 아닙니다. 만약 모든 문서에 너무 흔하게 나타나는 「그리고」, 「또한」 같은 단어라면 어떨까요? 이런 단어들은 아무리 문서 내에서 빈도가 높아도 특정 문서의 중요도를 판단하는 데 도움이 되지 않습니다. 반면 「인공지능」, 「머신러닝」, 「딥러닝」과 같은 특정 주제를 나타내는 단어들은 모든 문서에 흔히 등장하지 않고, 일부 문서에만 특별히 등장합니다. 이런 희귀한 단어들은 문서의 주제를 보다 명확하게 드러내는 단서가 되므로, BM25는 이런 단어에 더 높은 점수를 부여합니다. 이것이 바로 BM25가 중요하게 고려하는 두 번째 요소, **역문서 빈도(Inverse Document Frequency, IDF)**입니다.

역문서 빈도(IDF)를 조금 더 쉽게 표현하면, 특정 단어가 전체 문서 집합에서 얼마나 「희귀한지」를 수치화한 것입니다. IDF의 공식은 다음과 같습니다.

$$
\text{IDF} = \log\frac{N - n + 0.5}{n + 0.5}
$$

각 항목을 쉽게 설명하면 다음과 같습니다.

- $N$ : 전체 문서의 개수입니다.
- $n$ : 특정 단어가 나타나는 문서의 개수입니다.

즉, 특정 단어가 전체 문서 중에서 적은 수의 문서에서만 나타날수록, 다시 말해 더 희귀한 단어일수록 IDF 값은 높아지고, 이 단어가 해당 문서에 등장할 때의 중요도를 더 높게 판단하게 됩니다.

**문서 길이**  
마지막으로, BM25는 문서 길이라는 요소까지 고려하여 점수를 조정합니다. 같은 횟수로 등장한 단어라도 문서 길이에 따라 그 중요성이 달라지기 때문입니다. 예를 들어, 5000개의 단어로 이루어진 긴 문서에 「인공지능」이 5번 등장한 것과, 500개의 단어로 이루어진 짧은 문서에 같은 「인공지능」이 5번 등장한 것은 의미가 다릅니다. 긴 문서에서는 같은 횟수라도 그 중요성이 상대적으로 떨어질 수밖에 없습니다. BM25는 문서 길이를 기준으로, 긴 문서에 등장한 빈도의 가치를 낮추고, 짧은 문서에 등장한 빈도의 가치를 높이는 보정을 수행합니다.

**세가지 요소를 결합한 공식**

지금까지 설명한 세 가지 요소, 즉  
① 단어의 빈도(TF),  
② 단어가 얼마나 희귀한지(IDF),  
③ 문서의 길이  
를 모두 반영한 최종적인 BM25 공식은 다음과 같습니다.

$$
\text{BM25 점수} = \text{IDF} \times \frac{f \times (k_1 + 1)}{f + k_1 \times \left(1 - b + b \times \frac{L}{\text{avgL}}\right)}
$$

각 항목의 의미를 다시 간단히 정리하면 다음과 같습니다.

- $f$ : 특정 단어가 문서에서 등장한 빈도(횟수)
- $L$ : 문서의 길이(문서가 가진 전체 단어 수)
- $avgL$ : 문서 전체의 평균 길이
- $k_1$, $b$ : 빈도와 문서 길이를 조정하는 상수 (일반적으로 $k_1 = 1.2 \sim 2.0$, $b = 0.75$ 정도로 사용)
- IDF : 앞서 설명한 단어의 희귀성 지표(역문서 빈도)

정리하면, BM25는 사용자가 검색한 「인공지능」이라는 단어가 특정 문서 내에서 많이 등장할수록(빈도가 높을수록), 전체 문서 중 매우 적은 수의 문서에만 나타나는 희귀한 단어일수록(IDF가 클수록), 그리고 문서의 길이가 짧을수록(L이 avgL보다 작을수록), 더 높은 점수를 주어 해당 문서를 사용자가 원하는 검색 결과의 상위에 배치합니다.

In [ ]:
!pip -q install langchain openai tiktoken langchain-community rank_bm25 sentence-transformers chromadb pypdf langchain-huggingface langchain_openai

In [ ]:
import tiktoken
import openai
from typing import List, Dict, Optional, Any
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers  import BM25Retriever

In [ ]:
!wget https://wdr.ubion.co.kr/wowpass/img/event/gsat_170823/gsat_170823.pdf

In [ ]:
## pdf 파일로드 하고 쪼개기
loader = PyPDFLoader('https://wdr.ubion.co.kr/wowpass/img/event/gsat_170823/gsat_170823.pdf')
pages = loader.load_and_split()
print(len(pages))

In [ ]:
model_huggingface = HuggingFaceEmbeddings(model_name='BAAI/bge-m3')

In [ ]:
## chunk로 쪼개기
text_splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50)
texts = text_splitter.split_documents(pages)

In [ ]:
len(texts)

In [ ]:
print(texts[0])

## BM-25

In [ ]:
bm25_retriever = BM25Retriever.from_documents(texts)
bm25_retriever.k = 2

In [ ]:
# BM25 리트리버만 사용하는 경우
docs = bm25_retriever.invoke("삼성전자의 반도체 사업은 어떤 영역으로 구성되는가?")

for i in docs:
    print(i.metadata)
    print(":")
    print(i.page_content.replace('\n',' '))
    print(len(i.page_content.replace('\n',' ')))
    print("*"*30)

## 임베딩

In [ ]:
# 벡터 DB의 임베딩으로는 오픈소스 임베딩을 사용
chroma_vector = Chroma.from_documents(texts, model_huggingface)
chroma_retriever = chroma_vector.as_retriever(search_kwargs={'k':2})

In [ ]:
# 크로마 리트리버만 사용하는 경우
docs = chroma_retriever.invoke("삼성전자의 반도체 사업은 어떤 영역으로 구성되는가?")
for i in docs:
    print(i.metadata)
    print(":")
    print(i.page_content.replace('\n',' '))
    print(len(i.page_content.replace('\n',' ')))
    print("*"*30)

## 앙상블

### EnsembleRetriever 작동 과정 (상세 예시)

먼저 두 검색기가 있고, 각각 k=4개의 문서를 반환한다고 가정해보겠습니다:

#### 1. 개별 검색기 결과 (각 검색기의 상위 4개 문서)
```
BM25 검색기 결과:
- 문서 A: 점수 0.9
- 문서 B: 점수 0.8
- 문서 C: 점수 0.7
- 문서 D: 점수 0.6

임베딩 검색기 결과:
- 문서 A: 점수 0.95 (BM25와 중복)
- 문서 E: 점수 0.85
- 문서 F: 점수 0.75
- 문서 B: 점수 0.65 (BM25와 중복)
```
여기서 문서 A와 B는 두 검색기 모두에서 반환되었습니다.

#### 2. 가중치 적용 (각 검색기에 0.5씩 가중치 부여)

각 검색기의 점수에 가중치 0.5를 곱합니다:
```
BM25 검색기:
- 문서 A: 0.9 × 0.5 = 0.45
- 문서 B: 0.8 × 0.5 = 0.40
- 문서 C: 0.7 × 0.5 = 0.35
- 문서 D: 0.6 × 0.5 = 0.30

임베딩 검색기:
- 문서 A: 0.95 × 0.5 = 0.475
- 문서 E: 0.85 × 0.5 = 0.425
- 문서 F: 0.75 × 0.5 = 0.375
- 문서 B: 0.65 × 0.5 = 0.325
```


#### 3. 중복 문서 점수 합산

동일한 문서가 여러 검색기에서 나온 경우, 가중치가 적용된 점수들을 합산합니다:
```
- 문서 A: 0.45 (BM25) + 0.475 (임베딩) = 0.925
- 문서 B: 0.40 (BM25) + 0.325 (임베딩) = 0.725
- 문서 C: 0.35 (BM25만) = 0.35
- 문서 D: 0.30 (BM25만) = 0.30
- 문서 E: 0.425 (임베딩만) = 0.425
- 문서 F: 0.375 (임베딩만) = 0.375
```


#### 4. 점수 기준 정렬 및 최종 결과 반환
```
모든 문서를 최종 점수 기준으로 내림차순 정렬하고, 상위 k개(예: k=4)를 반환합니다:

최종 정렬:
1. 문서 A: 0.925 (두 검색기 모두에서 높은 점수)
2. 문서 B: 0.725 (두 검색기 모두에서 등장)
3. 문서 E: 0.425 (임베딩 검색기에서만)
4. 문서 F: 0.375 (임베딩 검색기에서만)
5. 문서 C: 0.35 (BM25 검색기에서만)
6. 문서 D: 0.30 (BM25 검색기에서만)

최종 반환 결과(k=4): 문서 A, B, E, F
```

### 핵심 포인트
1. **중복 문서 강화**: 두 검색기 모두에서 나온 문서(A, B)는 점수가 합산되어 순위가 높아집니다. 이는 여러 방식으로 관련성이 확인된 문서가 우선시됨을 의미합니다.

2. **다양한 결과 포함**: 각 검색기의 고유한 강점을 활용하여 키워드 매칭(BM25)과 의미적 유사성(임베딩) 모두에서 관련성 높은 문서를 포함합니다.

3. **가중치 영향**: 만약 BM25에 더 높은 가중치(예: 0.7)를 부여한다면, BM25 결과가 최종 순위에 더 큰 영향을 미치게 됩니다.

In [ ]:
class EnsembleRetriever:
    """
    BM25 + Dense retriever 앙상블.
    Reciprocal Rank Fusion (RRF) 알고리즘으로 점수 결합.
    """
    def __init__(self, retrievers: List, weights: Optional[List[float]] = None, c: int = 60, id_key: Optional[str] = None):
        assert len(retrievers) == len(weights), "retrievers와 weights 길이가 다릅니다."
        self.retrievers = retrievers
        self.weights = weights
        self.c = c
        self.id_key = id_key

    def _get_doc_id(self, doc: Document) -> str:
        """문서 중복 판단용 ID 추출"""
        if self.id_key and self.id_key in doc.metadata:
            return str(doc.metadata[self.id_key])
        return doc.page_content[:150]

    def _fuse_results(self, all_results: List[List[Document]]) -> List[Document]:
        """Reciprocal Rank Fusion 계산"""
        scores: Dict[str, Dict[str, Any]] = {}
        for i, docs in enumerate(all_results):
            weight = self.weights[i] if self.weights else 1.0
            for rank, doc in enumerate(docs):
                doc_id = self._get_doc_id(doc)
                score = weight / (rank + 1 + self.c)
                if doc_id not in scores:
                    scores[doc_id] = {"doc": doc, "score": score}
                else:
                    scores[doc_id]["score"] += score

        sorted_docs = sorted(scores.values(), key=lambda x: x["score"], reverse=True)
        return [item["doc"] for item in sorted_docs]

    def invoke(self, query: str, k: int = 4) -> List[Document]:
        """모든 retriever에서 검색 후 점수 융합"""
        all_results = []
        for retriever in self.retrievers:
            if hasattr(retriever, "get_relevant_documents"):
                docs = retriever.get_relevant_documents(query)
            elif hasattr(retriever, "invoke"):
                docs = retriever.invoke(query)
            else:
                raise AttributeError(f"{retriever} does not support retrieval.")
            all_results.append(docs[:k])

        fused_docs = self._fuse_results(all_results)
        return fused_docs[:k]

In [ ]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.35, 0.65]  # 키워드 35%, 의미 검색 65%
)

docs = ensemble_retriever.invoke("삼성전자의 반도체 사업은 어떤 영역으로 구성되는가?")

In [ ]:
# 문서의 개수는 총 4개이다.
len(docs)

## 앙상블 결과

In [ ]:
for i in docs:
    print(i.metadata)
    print(":")
    print(i.page_content.replace('\n',' '))
    print(len(i.page_content.replace('\n',' ')))
    print("*"*30)

## 하이브리드 서치

### 두 검색 방식의 차이점: 간단 예시

**문서 목록:**
- A: "삼성전자 반도체"
- B: "삼성전자 스마트폰"
- C: "반도체 기술"
- D: "삼성 신제품"

**검색어:** "삼성 반도체"

**각 검색기의 결과:**
- BM25 결과: A, C, B, D (키워드 일치 순)
- 임베딩 결과: A, D, B, C (의미 유사도 순)

### 1. hybrid_search (라운드 로빈 방식)
번갈아가면서 결과를 가져옵니다:
1. BM25 첫번째: A
2. 임베딩 첫번째: A (이미 있으므로 건너뜀)
3. BM25 두번째: C
4. 임베딩 두번째: D
5. BM25 세번째: B

**최종 결과:** A, C, D (순서대로)

### 2. (앞에서 정리한) EnsembleRetriever (점수 합산 방식)
점수를 계산하고 합산합니다:
- A: BM25(0.9) + 임베딩(0.95) = 1.85
- B: BM25(0.7) + 임베딩(0.8) = 1.5
- C: BM25(0.8) + 임베딩(0.7) = 1.5
- D: BM25(0.6) + 임베딩(0.9) = 1.5

**최종 결과:** A, B/C/D (점수순)

### 핵심 차이:
- **hybrid_search**: 다양한 검색 방식의 결과를 번갈아 포함시켜 다양성 중시
- **EnsembleRetriever**: 여러 검색기에서 모두 높은 점수를 받은 문서 우선 (정확도 중시)

In [ ]:
from langchain_core.documents import Document
from typing import List, Set

# 두 검색 방법을 별도로 실행하고 결과 통합
def hybrid_search(query: str, k: int = 5) -> List[Document]:
    """
    BM25와 임베딩 기반 검색을 결합한 하이브리드 검색 구현

    이 함수는 키워드 기반 검색(BM25)과 의미 기반 검색(임베딩)을 모두 수행하고
    결과를 번갈아가며 통합하여 다양한 관점의 검색 결과를 제공합니다.

    Args:
        query (str): 검색 쿼리
        k (int): 반환할 최대 문서 수

    Returns:
        List[Document]: 중복이 제거된 통합 검색 결과
    """
    # BM25 검색 결과 가져오기 (키워드 기반 검색)
    bm25_results = bm25_retriever.invoke(query)

    # 임베딩 검색 결과 가져오기 (의미 기반 검색)
    embedding_results = chroma_retriever.invoke(query)

    # 결과 통합 및 중복 제거를 위한 변수 초기화
    seen_contents: Set[str] = set()  # 이미 처리된 문서 내용을 추적
    hybrid_results: List[Document] = []  # 최종 결과를 저장할 리스트

    # 두 결과 세트를 번갈아가며 통합 (라운드 로빈 방식)
    for i in range(max(len(bm25_results), len(embedding_results))):
        # BM25 결과 처리 (있는 경우)
        if i < len(bm25_results):
            doc = bm25_results[i]
            # 중복 문서 확인 및 추가
            if doc.page_content not in seen_contents:
                seen_contents.add(doc.page_content)
                hybrid_results.append(doc)

        # 임베딩 검색 결과 처리 (있는 경우)
        if i < len(embedding_results):
            doc = embedding_results[i]
            # 중복 문서 확인 및 추가
            if doc.page_content not in seen_contents:
                seen_contents.add(doc.page_content)
                hybrid_results.append(doc)

        # 요청한 수(k)만큼 문서를 찾았으면 중단
        if len(hybrid_results) >= k:
            break

    # 최대 k개의 결과 반환
    return hybrid_results[:k]

In [ ]:
# 사용 예시 - 실제 쿼리로 하이브리드 검색 실행
results = hybrid_search("삼성전자의 반도체 사업은 어떤 영역으로 구성되는가?", k=5)

In [ ]:
results

## 답변 얻기

In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Colab 왼쪽의 Secrets에 OPENAI_API_KEY를 등록하세요.")

os.environ["OPENAI_API_KEY"] = api_key


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4.1", temperature=0)

prompt = ChatPromptTemplate.from_template("""
아래 문서만 근거로 답하세요. 먼저 한 문장으로 결론을 쓰고, 근거를 항목별로 정리하세요. 문서에 없는 내용은 추측하지 마세요.

질문: {question}

참고 문서:
{context}

답변:
""")

def ask_question(question: str, retriever):
    # 질문으로 유사도 검색
    docs = retriever.invoke(question)
    # 검색 문서를 1개의 문자열로 연결
    context = "\n\n".join(doc.page_content for doc in docs)
    # 프롬프트 템플릿에 질문하고 검색 결과 전달
    messages = prompt.format_messages(question=question, context=context)
    # 답변
    result = llm.invoke(messages)
    return result.content

In [ ]:
question = "삼성전자의 반도체 사업은 어떤 영역으로 구성되는가?"

In [ ]:
print("\n[Ensemble 검색 결과]")
print(ask_question(question, ensemble_retriever))

In [ ]:
print("\n[Chroma 검색 결과]")
print(ask_question(question, chroma_retriever))

In [ ]:
print("\n[BM25 검색 결과]")
print(ask_question(question, bm25_retriever))

하이브리드 검색을 RetrievalQA에 사용하려면 먼저 커스텀 Retriever 클래스를 만들어야 합니다.

In [ ]:
class HybridRetriever:
    def __init__(self, bm25_retriever, chroma_retriever, k: int = 5):
        self.bm25_retriever = bm25_retriever
        self.chroma_retriever = chroma_retriever
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        bm25_results = self.bm25_retriever.invoke(query)
        chroma_results = self.chroma_retriever.invoke(query)

        seen_contents: Set[str] = set()
        hybrid_results: List[Document] = []

        for i in range(max(len(bm25_results), len(chroma_results))):
            if i < len(bm25_results):
                doc = bm25_results[i]
                if doc.page_content not in seen_contents:
                    seen_contents.add(doc.page_content)
                    hybrid_results.append(doc)

            if i < len(chroma_results):
                doc = chroma_results[i]
                if doc.page_content not in seen_contents:
                    seen_contents.add(doc.page_content)
                    hybrid_results.append(doc)

            if len(hybrid_results) >= self.k:
                break

        return hybrid_results[:self.k]

In [ ]:
hybrid_retriever = HybridRetriever(
    bm25_retriever=bm25_retriever,
    chroma_retriever=chroma_retriever,
    k=5
)

In [ ]:
print("\n[Hybrid 검색 결과]")
print(ask_question(question, hybrid_retriever))